# Gold: patient_360

Builds `dbw_caresync_dev.gold.patient_360` (one row per patient) from `silver.patients`, enriched with hospital, insurance provider and aggregated lab-result metrics, per STTM `sttm_silver_to_gold.xlsx` (sheet `patient_360`).

* **Driving table:** `silver.patients` (all patients, no filters)
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`; `silver.insurance_providers` on `insurance_provider_id`; `silver.lab_results` aggregated by `patient_id`
* **Load:** FULL — `CREATE OR REPLACE TABLE ... AS SELECT`
* **Schema note:** `silver.patients.date_of_birth` is `timestamp` (all values at midnight); it is cast to `date` to match the target type.

In [0]:
%sql
-- FULL rebuild: one row per patient (driving table silver.patients, LEFT JOINs only)
CREATE OR REPLACE TABLE dbw_caresync_dev.gold.patient_360 AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                  AS total_tests,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count,
    MAX(test_date)                            AS last_test_date
  FROM dbw_caresync_dev.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                   AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                        AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name                                                                 AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                         AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                                 AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                                 AS critical_test_count,
  l.last_test_date
FROM dbw_caresync_dev.silver.patients p
LEFT JOIN dbw_caresync_dev.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN dbw_caresync_dev.silver.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id

In [0]:
# Runs only after the CREATE OR REPLACE TABLE cell above succeeds; returns the Gold row count to ADF (runOutput)
target_table = "dbw_caresync_dev.gold.patient_360"
record_count = spark.table(target_table).count()
print(f"SUCCESS: {record_count} record(s) in {target_table}")
dbutils.notebook.exit(str(record_count))